# 01. Source Inventory & Coverage Scoping
## Corporate Carbon Footprint Tracker (Task EDA-1)
### Deliverable 1: Data Collection, Extraction, Cleaning & Documentation

---

### Purpose
This notebook defines the corporate cohort, sector classifications, SEC EDGAR Central Index Keys (CIKs), and public disclosure report URLs for **Deliverable 1**.
It establishes the traceable registry for downstream API collection (`02_api_collection.ipynb`) and programmatic PDF extraction (`03_pdf_extraction.ipynb`).

### Scope & Criteria
- **Reporting Period**: FY2019 – FY2023 (5-year historical window).
- **Target Entities**: 12 high-impact corporations across Technology, Energy, Industrial/Automotive, and Consumer sectors.
- **Inclusion Standards**:
  1. Mandatory SEC 10-K regulatory filings for audited financial revenues.
  2. Public standalone Sustainability / ESG / Environmental Progress reports disclosing Scope 1, Scope 2 (market & location), and Scope 3 emissions.
  3. Auditable primary documents with third-party assurance statements (e.g., Deloitte, PwC, Apex).


In [ ]:
import os
import pandas as pd
import numpy as np

# Verify paths
RAW_DATA_DIR = os.path.join("..", "data", "raw")
INVENTORY_CSV = os.path.join(RAW_DATA_DIR, "source_inventory.csv")

print(f"Loading inventory from: {INVENTORY_CSV}")
assert os.path.exists(INVENTORY_CSV), f"Missing inventory file: {INVENTORY_CSV}"


## 1. Load Corporate Cohort & Disclosures Registry

In [ ]:
df_inventory = pd.read_csv(INVENTORY_CSV)
print(f"Total Companies Registered: {len(df_inventory)}")
df_inventory[['company_id', 'company_name', 'ticker', 'cik', 'sector', 'reporting_years_covered']]


## 2. Sectoral Breakdown
Evaluating the distribution of target companies across key economic sectors to ensure cross-industry analytical validity.


In [ ]:
sector_counts = df_inventory['sector'].value_counts().reset_index()
sector_counts.columns = ['Sector', 'Company Count']
print("Sector Distribution:")
display(sector_counts)


## 3. Coverage Matrix: Company × Reporting Years
Constructing the target observation matrix across the 5-year timeline (2019–2023).


In [ ]:
years = [2019, 2020, 2021, 2022, 2023]
matrix_data = []

for _, row in df_inventory.iterrows():
    for y in years:
        matrix_data.append({
            'company_id': row['company_id'],
            'company_name': row['company_name'],
            'ticker': row['ticker'],
            'sector': row['sector'],
            'reporting_year': y,
            'status': 'Planned Extraction'
        })

df_matrix = pd.DataFrame(matrix_data)
print(f"Total Planned Company-Year Observations: {len(df_matrix)}")
pivot_summary = df_matrix.pivot(index=['company_id', 'ticker'], columns='reporting_year', values='status')
display(pivot_summary)


## 4. Source URL & Retrieval Protocol Verification
Documenting the access protocols, endpoints, and third-party assurance notes for every source.


In [ ]:
for idx, row in df_inventory.iterrows():
    print(f"[{row['company_id']}] {row['company_name']} ({row['ticker']})")
    print(f"  Sector: {row['sector']} | CIK: {row['cik']}")
    print(f"  SEC Endpoint: https://data.sec.gov/api/xbrl/companyfacts/CIK{str(row['cik']).zfill(10)}.json")
    print(f"  Primary ESG Report: {row['primary_report_title']}")
    print(f"  Report URL: {row['report_url'][:80]}...")
    print(f"  Assurance / Notes: {row['access_notes']}")
    print("-" * 80)


## 5. Summary & Next Steps
- **Registry Established**: 12 companies across 4 major sectors.
- **Total Expected Observations**: 60 company-year records.
- **Handoff**:
  - Proceed to **`02_api_collection.ipynb`** to query the SEC EDGAR API for audited 10-K revenues.
  - Proceed to **`03_pdf_extraction.ipynb`** to download primary ESG PDFs and parse Scope 1–3 tables.
